In [0]:
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"""


CREATE OR REPLACE TABLE {catalog}.gold.fact_bookings
(
    booking_segment_id    STRING        COMMENT 'Surrogate / natural key that uniquely identifies one booking-segment line',
    booking_id            STRING        COMMENT 'FK to the parent reservation (dim_booking / booking header)',
    passenger_id          STRING        COMMENT 'FK to passenger dimension',
    flight_id             STRING        COMMENT 'FK to flight dimension',
    segment_id            STRING        COMMENT 'FK to flight-segment dimension (origin-destination pair)',
    aircraft_id           STRING        COMMENT 'FK to aircraft dimension',
    route_id              STRING        COMMENT 'FK to route dimension',

    -- Flight-segment context (denormalised for convenience / query ergonomics)
    flight_number         STRING        COMMENT 'Carrier flight number, e.g. DL123',
    carrier_code          STRING        COMMENT 'Operating carrier IATA code',
    origin_airport_code   STRING        COMMENT 'IATA code of the departure airport',
    dest_airport_code     STRING        COMMENT 'IATA code of the arrival airport',
    departure_ts          TIMESTAMP     COMMENT 'Scheduled departure timestamp (local)',
    arrival_ts            TIMESTAMP     COMMENT 'Scheduled arrival timestamp (local)',

    -- Booking attributes
    booking_date          DATE          COMMENT 'Date the booking was made',
    cabin_class           STRING        COMMENT 'Economy / Premium Economy / Business / First',
    fare_class            STRING        COMMENT 'Fare basis code or booking class',
    fare_basis_code       STRING        COMMENT 'Detailed fare-basis code',
    seat_number           STRING        COMMENT 'Assigned seat number (nullable until check-in)',
    booking_status        STRING        COMMENT 'Confirmed / Cancelled / Waitlisted / Checked-in / No-show',
    passenger_type        STRING        COMMENT 'Adult / Child / Infant',
    trip_type             STRING        COMMENT 'One-way / Round-trip / Multi-city',

    -- Measures (additive facts)
    base_fare_amount      DECIMAL(12,2) COMMENT 'Base fare paid, excluding taxes and fees',
    tax_amount            DECIMAL(12,2) COMMENT 'Government and airport taxes',
    fee_amount            DECIMAL(12,2) COMMENT 'Carrier-imposed fees (fuel, service, etc.)',
    total_amount          DECIMAL(12,2) COMMENT 'Total ticketed amount (base + tax + fee)',
    currency_code         STRING        COMMENT 'ISO-4217 currency of all monetary measures',
    miles_earned         DECIMAL(12,2) COMMENT 'Frequent-flyer miles awarded for this segment',

    -- Additional booking attributes (from silver source)
    booking_priority     STRING        COMMENT 'Priority level of the booking',
    agent                STRING        COMMENT 'Booking agent identifier',
    comment              STRING        COMMENT 'Booking comment / notes',

    -- Audit / lineage
    source_system         STRING        COMMENT 'Originating source system identifier',
    ingested_at           TIMESTAMP     COMMENT 'Timestamp the row was loaded into silver',
    created_at            TIMESTAMP     COMMENT 'Row insert time in the gold table',
    updated_at            TIMESTAMP     COMMENT 'Row last-update time in the gold table'
)
USING DELTA
COMMENT 'Gold fact table: one row per flight-segment booking line'
CLUSTER BY (booking_id, segment_id)
""")

spark.sql(f"""
INSERT INTO {catalog}.gold.fact_bookings
SELECT
    /*  Surrogate key: concatenate booking-id and passenger-id so
        every row is unique at the booking-passenger grain.  */
    CONCAT_WS('~~', b.bk_bookingid, b.bk_passengerid)                AS booking_segment_id,

    b.bk_bookingid                                                   AS booking_id,
    b.bk_passengerid                                                 AS passenger_id,
    NULL                                                             AS flight_id,
    NULL                                                             AS segment_id,
    NULL                                                             AS aircraft_id,
    NULL                                                             AS route_id,

    NULL                                                             AS flight_number,
    NULL                                                             AS carrier_code,
    NULL                                                             AS origin_airport_code,
    NULL                                                             AS dest_airport_code,
    CAST(b.bk_flightdate AS TIMESTAMP)                               AS departure_ts,
    NULL                                                             AS arrival_ts,

    CAST(b.bk_bookingdate AS DATE)                                   AS booking_date,
    NULL                                                             AS cabin_class,
    NULL                                                             AS fare_class,
    NULL                                                             AS fare_basis_code,
    NULL                                                             AS seat_number,
    b.bk_status                                                      AS booking_status,
    NULL                                                             AS passenger_type,
    NULL                                                             AS trip_type,

    NULL                                                             AS base_fare_amount,
    NULL                                                             AS tax_amount,
    NULL                                                             AS fee_amount,
    b.bk_totalprice                                                  AS total_amount,
    NULL                                                             AS currency_code,
    NULL                                                             AS miles_earned,

    b.bk_bookingpriority                                             AS booking_priority,
    b.bk_agent                                                       AS agent,
    b.bk_comment                                                     AS comment,

    NULL                                                             AS source_system,
    NULL                                                             AS ingested_at,
    current_timestamp()                                              AS created_at,
    current_timestamp()                                              AS updated_at
FROM {catalog}.silver.booking  b
WHERE b.bk_bookingid     IS NOT NULL
  AND b.bk_passengerid    IS NOT NULL;
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
# Read schema from silver.passenger to get exact column names and types
src_df = spark.read.table(f"{catalog}.silver.passenger")
fields = src_df.schema.fields

# Identify the passenger ID column for clustering and deduplication
id_col = next(
    (f.name for f in fields
     if "passengerid" in f.name.lower() or "passenger_id" in f.name.lower()),
    fields[0].name  # fallback to first column
)

# Helper: derive a human-readable comment from a column name
def col_comment(name: str) -> str:
    return name.replace("_", " ").strip().title()

# Build CREATE TABLE DDL with a comment on every column
col_defs = []
for f in fields:
    col_defs.append(f"    {f.name}  {f.dataType.simpleString()}  COMMENT '{col_comment(f.name)}'")

ddl = f"""CREATE OR REPLACE TABLE {catalog}.gold.dim_passenger
(
{',\n'.join(col_defs)}
)
USING DELTA
COMMENT 'Gold dimension table: one row per passenger'
CLUSTER BY ({id_col})"""

spark.sql(ddl)
print(f"\nCreated {catalog}.gold.dim_passenger clustered by {id_col}")

# Deduplicate: keep exactly one row per passenger
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy(id_col).orderBy(id_col)
deduped = (
    src_df
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

col_list = ", ".join([f.name for f in fields])
deduped.createOrReplaceTempView("_dim_passenger_src")

spark.sql(f"""
INSERT INTO {catalog}.gold.dim_passenger ({col_list})
SELECT {col_list}
FROM _dim_passenger_src
""")

print(f"Inserted deduplicated rows into {catalog}.gold.dim_passenger")
print(f"Row count: {spark.read.table(f'{catalog}.gold.dim_passenger').count():,}")


Created dev.gold.dim_passenger clustered by pa_passengerid
Inserted deduplicated rows into dev.gold.dim_passenger
Row count: 1,500


In [0]:
# Read schema from silver.route to get exact column names and types
src_df = spark.read.table(f"{catalog}.silver.route")
fields = src_df.schema.fields

# Identify the route ID column for clustering and deduplication
id_col = next(
    (f.name for f in fields
     if "routeid" in f.name.lower() or "route_id" in f.name.lower()),
    fields[0].name  # fallback to first column
)

# Helper: derive a human-readable comment from a column name
def col_comment(name: str) -> str:
    return name.replace("_", " ").strip().title()

# Build CREATE TABLE DDL with a comment on every column
col_defs = []
for f in fields:
    col_defs.append(f"    {f.name}  {f.dataType.simpleString()}  COMMENT '{col_comment(f.name)}'")

ddl = f"""CREATE OR REPLACE TABLE {catalog}.gold.dim_route
(
{',\n'.join(col_defs)}
)
USING DELTA
COMMENT 'Gold dimension table: one row per route'
CLUSTER BY ({id_col})"""

spark.sql(ddl)
print(f"\nCreated {catalog}.gold.dim_route clustered by {id_col}")

# Deduplicate: keep exactly one row per route
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy(id_col).orderBy(id_col)
deduped = (
    src_df
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

col_list = ", ".join([f.name for f in fields])
deduped.createOrReplaceTempView("_dim_route_src")

spark.sql(f"""
INSERT INTO {catalog}.gold.dim_route ({col_list})
SELECT {col_list}
FROM _dim_route_src
""")

print(f"Inserted deduplicated rows into {catalog}.gold.dim_route")
print(f"Row count: {spark.read.table(f'{catalog}.gold.dim_route').count():,}")


Created dev.gold.dim_route clustered by ro_routeid
Inserted deduplicated rows into dev.gold.dim_route
Row count: 100


In [0]:
# Read schema from silver.airline to get exact column names and types
src_df = spark.read.table(f"{catalog}.silver.airline")
fields = src_df.schema.fields

# Identify the airline ID column for clustering and deduplication
id_col = next(
    (f.name for f in fields
     if "airlineid" in f.name.lower() or "airline_id" in f.name.lower()),
    fields[0].name  # fallback to first column
)

# Helper: derive a human-readable comment from a column name
def col_comment(name: str) -> str:
    return name.replace("_", " ").strip().title()

# Build CREATE TABLE DDL with a comment on every column
col_defs = []
for f in fields:
    col_defs.append(f"    {f.name}  {f.dataType.simpleString()}  COMMENT '{col_comment(f.name)}'")

ddl = f"""CREATE OR REPLACE TABLE {catalog}.gold.dim_airline
(
{',\n'.join(col_defs)}
)
USING DELTA
COMMENT 'Gold dimension table: one row per airline'
CLUSTER BY ({id_col})"""

spark.sql(ddl)
print(f"\nCreated {catalog}.gold.dim_airline clustered by {id_col}")

# Deduplicate: keep exactly one row per airline
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy(id_col).orderBy(id_col)
deduped = (
    src_df
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

col_list = ", ".join([f.name for f in fields])
deduped.createOrReplaceTempView("_dim_airline_src")

spark.sql(f"""
INSERT INTO {catalog}.gold.dim_airline ({col_list})
SELECT {col_list}
FROM _dim_airline_src
""")

print(f"Inserted deduplicated rows into {catalog}.gold.dim_airline")
print(f"Row count: {spark.read.table(f'{catalog}.gold.dim_airline').count():,}")


Created dev.gold.dim_airline clustered by al_airlineid
Inserted deduplicated rows into dev.gold.dim_airline
Row count: 10


In [0]:
# Read schemas from silver.nation and silver.region
nation_df = spark.read.table(f"{catalog}.silver.nation")
region_df = spark.read.table(f"{catalog}.silver.region")

nation_fields = nation_df.schema.fields
region_fields = region_df.schema.fields

# Identify the nation ID column for clustering and deduplication
nation_id_col = next(
    (f.name for f in nation_fields
     if "nationid" in f.name.lower() or "nation_id" in f.name.lower()),
    nation_fields[0].name
)

# Identify the region FK column in nation (the join key)
region_fk_col = next(
    (f.name for f in nation_fields
     if f.name.lower() not in (nation_id_col.lower(),)
     and ("regionid" in f.name.lower() or "region_id" in f.name.lower())),
    None
)

# Identify the region ID column in the region table
region_id_col = next(
    (f.name for f in region_fields
     if "regionid" in f.name.lower() or "region_id" in f.name.lower()),
    region_fields[0].name
)

print(f"Nation ID col : {nation_id_col}")
print(f"Region FK col : {region_fk_col}")
print(f"Region ID col : {region_id_col}")

# Helper: derive a human-readable comment from a column name
def col_comment(name: str) -> str:
    return name.replace("_", " ").strip().title()

# Build the list of output columns: all nation columns + region columns (prefixed)
output_col_defs = []
for f in nation_fields:
    output_col_defs.append(
        (f.name, f"{f.dataType.simpleString()}", col_comment(f.name))
    )

for f in region_fields:
    alias = f"parent_region_{f.name}"
    # Avoid duplicate column names
    if alias not in [c[0] for c in output_col_defs]:
        output_col_defs.append(
            (alias, f"{f.dataType.simpleString()}", f"Parent region {col_comment(f.name)}")
        )

# Build CREATE TABLE DDL
col_def_sql = []
for name, dtype, comment in output_col_defs:
    col_def_sql.append(f"    {name}  {dtype}  COMMENT '{comment}'")

ddl = f"""CREATE OR REPLACE TABLE {catalog}.gold.dim_geography
(
{',\n'.join(col_def_sql)}
)
USING DELTA
COMMENT 'Gold dimension table: one row per nation, including its parent region'
CLUSTER BY ({nation_id_col})"""

spark.sql(ddl)
print(f"\nCreated {catalog}.gold.dim_geography clustered by {nation_id_col}")

# Join nation with region on the region FK, then deduplicate by nation ID
from pyspark.sql import functions as F
from pyspark.sql.window import Window

if region_fk_col is not None:
    joined_df = nation_df.join(
        region_df,
        nation_df[region_fk_col] == region_df[region_id_col],
        "left"
    )
else:
    # No region FK found – just use nation columns, region columns will be NULL
    joined_df = nation_df
    for f in region_fields:
        joined_df = joined_df.withColumn(f"parent_region_{f.name}", F.lit(None).cast(f.dataType.simpleString()))

# Rename region columns to add parent_region_ prefix
for f in region_fields:
    if f.name != region_id_col or True:  # rename all region columns
        alias = f"parent_region_{f.name}"
        if alias not in [c.name for c in joined_df.schema.fields] and f.name in [c.name for c in joined_df.schema.fields]:
            joined_df = joined_df.withColumnRenamed(f.name, alias)

# Deduplicate: keep exactly one row per nation
w = Window.partitionBy(nation_id_col).orderBy(nation_id_col)
deduped = (
    joined_df
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# Select only the output columns in the correct order
output_col_names = [c[0] for c in output_col_defs]
deduped = deduped.select(*output_col_names)

deduped.createOrReplaceTempView("_dim_geography_src")

col_list = ", ".join(output_col_names)
spark.sql(f"""
INSERT INTO {catalog}.gold.dim_geography ({col_list})
SELECT {col_list}
FROM _dim_geography_src
""")

print(f"Inserted deduplicated rows into {catalog}.gold.dim_geography")
print(f"Row count: {spark.read.table(f'{catalog}.gold.dim_geography').count():,}")

Nation ID col : n_nationkey
Region FK col : None
Region ID col : r_regionkey

Created dev.gold.dim_geography clustered by n_nationkey
Inserted deduplicated rows into dev.gold.dim_geography
Row count: 25


In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.vw_annual_booking_kpis
(
    booking_year        COMMENT 'Year of the booking date',
    net_revenue         COMMENT 'Net revenue (ticket price after discount)',
    distinct_bookings   COMMENT 'Count of distinct bookings',
    active_passengers   COMMENT 'Count of distinct passengers with bookings',
    avg_booking_value   COMMENT 'Average booking value (net revenue / distinct bookings)'
)
COMMENT 'Annual booking KPIs: net revenue, distinct bookings, active passengers, and average booking value'
AS
SELECT
    YEAR(bk_bookingdate)                                    AS booking_year,

    -- Net revenue (before tax)
    SUM(bk_totalprice)                                      AS net_revenue,

    -- Distinct bookings
    COUNT(DISTINCT bk_bookingid)                             AS distinct_bookings,

    -- Active passengers (passengers with at least one booking in the year)
    COUNT(DISTINCT bk_passengerid)                           AS active_passengers,

    -- Average booking value = net revenue / distinct bookings
    SUM(bk_totalprice)
        / COUNT(DISTINCT bk_bookingid)                       AS avg_booking_value

FROM {catalog}.silver.booking
WHERE bk_bookingdate IS NOT NULL
  AND bk_bookingid    IS NOT NULL
GROUP BY YEAR(bk_bookingdate)
""")

print(f"Created view {catalog}.gold.vw_annual_booking_kpis")
print("View is ready for querying.")
spark.read.table(f"{catalog}.gold.vw_annual_booking_kpis").display()

Created view dev.gold.vw_annual_booking_kpis
View is ready for querying.


booking_year,net_revenue,distinct_bookings,active_passengers,avg_booking_value
1994,1.2272592100000009E7,1611,990,7617.996337678466
1998,1.2556859719999995E7,1652,997,7601.004673123483
1997,1.2111764249999985E7,1612,990,7513.501395781628
1995,1.1439058799999999E7,1522,941,7515.8073587385015
1996,1.2170276740000002E7,1633,1003,7452.710802204533
